# Day 2 · 01. 하이브리드 · 앙상블 검색

## 학습 목표
- BM25의 점수식(TF, IDF, 문서 길이 정규화)을 설명하고 `rank_bm25`로 구현할 수 있다.
- 한국어 BM25에서 **형태소 분석 토크나이저**(kiwi)가 왜 필수인지 공백 분할과 비교해 보일 수 있다.
- Dense(FAISS) + Sparse(BM25) 결과를 **RRF**로 결합하는 `EnsembleRetriever`를 구성할 수 있다.

## 핵심 키워드
BM25 · IDF · 길이 정규화 · RRF(Reciprocal Rank Fusion) · EnsembleRetriever · Sparse-Dense Hybrid

## 이 노트북의 위치
프로덕션 RAG의 표준 패턴은 **2-stage retrieval** 이다.

```
1단계 (이 노트북)                 2단계 (다음 노트북)
Hybrid (Dense + BM25, RRF)  ─▶  Cross-Encoder Reranker
└─ "넓게 후보를 모은다"            └─ "정답을 꼭대기에 올린다"
       recall 확보                      precision 확보
```

Anthropic의 Contextual Retrieval 실험(2024)도 동일한 구조로 실패율을 **67% 감소**시켰다 (dense 단독 대비). 이 노트북에서는 **1단계(Hybrid)** 를 구성하고, 이어지는 `02_reranking.ipynb` 에서 **2단계(Reranker)** 를 얹어 완성형 파이프라인을 만든다.


In [ ]:
# 공통 세팅: common 헬퍼를 사용하기 위해 repo 루트를 sys.path에 추가
import sys, os
sys.path.insert(0, '..')

from common import get_chat_model, get_embeddings, provider_badge
from common.ko_tokenizer import tokenize_ko
from common.loaders import load_any

print(provider_badge())


## 1. BM25가 뭐고, 왜 쓰는가?

BM25는 **"질문에 들어있는 단어가 어떤 문서에 많이/강하게 등장하는가"** 를 점수로 매기는 고전적 검색 알고리즘이다. 구글 이전부터 쓰이던 방식이고, 지금도 엘라스틱서치의 기본 랭킹 함수다.

임베딩 검색(dense)과 다른 점은 이것 하나다.
- **Dense 검색**: "뜻이 비슷한가" (의미 기반)
- **BM25 검색**: "같은 단어가 들어있는가" (단어 일치 기반)

그래서 `"전자금융감독규정 제13조"` 같은 **고유명사·번호·전문용어**를 찾을 때는 BM25가 압도적으로 유리하다. 임베딩은 "비슷한 뜻"을 찾다가 엉뚱한 조항을 끌어올 수 있기 때문이다.

### BM25가 점수를 매기는 세 가지 직관

BM25 점수는 결국 아래 세 가지 상식을 수식으로 정리한 것이다.

**① 질문에 있는 단어가 문서에 많이 나올수록 관련성이 높다 (TF, 단어 빈도)**
- "전자금융"이 10번 나온 문서가 1번 나온 문서보다 관련성이 높다.
- 단, 100번 → 1000번으로 늘어도 관련도가 10배 오르진 않는다. 그래서 BM25는 **빈도가 커지면 점수 증가를 둔화**시킨다 (포화).

**② 아무 문서에나 나오는 흔한 단어는 덜 중요하다 (IDF, 희귀도)**
- "~이다", "있다" 같이 모든 문서에 나오는 단어는 판별력이 0에 가깝다.
- 반대로 "전자금융감독규정"처럼 **몇몇 문서에만 등장하는 희귀 단어**가 더 강한 증거가 된다.
- IDF는 "전체 $N$개 문서 중 이 단어가 $n_t$개에만 나왔다"를 $\log$로 찍는다.

**③ 긴 문서는 단어가 많이 나오는 게 당연하니 보정해준다 (길이 정규화)**
- 100자짜리 문서에서 "전자금융"이 5번 나오는 것 ≠ 10,000자짜리 문서에서 5번 나오는 것.
- 그래서 문서 길이가 **평균보다 길면 점수를 깎고, 짧으면 점수를 올린다**.

### 수식으로 보면

세 직관을 합친 것이 BM25 점수식이다. 당장 외울 필요는 없고, "어떤 조각이 무엇을 의미하는지"만 감 잡으면 충분하다.

$$\text{score}(q, d) = \sum_{t \in q} \underbrace{\text{IDF}(t)}_{\text{② 희귀도}} \cdot \frac{\overbrace{f(t,d)}^{\text{① 빈도}}\,(k_1+1)}{f(t,d) + k_1 \left(1 - b + b \cdot \underbrace{\frac{|d|}{\text{avgdl}}}_{\text{③ 길이 보정}}\right)}$$

- $f(t,d)$: 문서 $d$에서 단어 $t$가 등장한 횟수
- $|d|, \text{avgdl}$: 이 문서의 길이 / 전체 문서 평균 길이
- $k_1 \approx 1.5$: 빈도가 늘어날 때 점수가 얼마나 빨리 포화될지 (클수록 천천히 포화)
- $b \approx 0.75$: 길이 보정을 얼마나 세게 할지 (0이면 길이 무시, 1이면 최대로 보정)

> 💡 **한 줄 요약**: "**희귀한 단어**가 **짧은 문서**에 **여러 번** 나올수록 점수가 높다." 그 이상도 이하도 아니다.


## 2. 한국어 BM25: 토크나이저가 성능을 좌우한다

영어 BM25는 공백 분할만 해도 잘 동작하지만, 한국어는 교착어라서 조사·어미가 붙어 다닌다. "전자금융거래는" ≠ "전자금융거래가" → 단순 공백 분할은 이 둘을 다른 토큰으로 본다.

In [ ]:
# 코퍼스 로드 및 청킹
docs = load_any('../data/corpus_ko.txt')
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
chunks = splitter.split_documents(docs)
texts = [c.page_content for c in chunks]
print(f'청크 수: {len(texts)}')
print('예시:', texts[0][:80])


In [ ]:
# 두 가지 토크나이저 비교
sample = '전자금융거래는 자동화된 방식으로 이용하는 거래를 말한다.'
print('공백 분할  :', sample.split())
print('kiwi(tokenize_ko):', tokenize_ko(sample))


In [ ]:
from rank_bm25 import BM25Okapi

# (A) 공백 분할 기반 BM25
corpus_ws = [t.split() for t in texts]
bm25_ws = BM25Okapi(corpus_ws)

# (B) kiwi 형태소 분석 기반 BM25
corpus_ko = [tokenize_ko(t) for t in texts]
bm25_ko = BM25Okapi(corpus_ko)

def top_k(bm25, tokenize, query, k=3):
    q = tokenize(query)
    scores = bm25.get_scores(q)
    idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    return [(scores[i], texts[i][:60]) for i in idx]

query = '전자금융거래가 무엇인지 설명해줘'
print('[공백 분할]')
for s, t in top_k(bm25_ws, str.split, query):
    print(f'  {s:.3f} | {t}')
print('\n[kiwi 형태소]')
for s, t in top_k(bm25_ko, tokenize_ko, query):
    print(f'  {s:.3f} | {t}')


> **관찰**: 공백 분할은 "전자금융거래가"와 "전자금융거래는"을 다른 토큰으로 인식해 매칭이 실패하거나 점수가 낮게 나온다. kiwi는 어간 "전자금융거래"를 동일 토큰으로 추출해 정확히 매칭한다.


## 3. Reciprocal Rank Fusion (RRF): 두 검색 결과를 어떻게 합칠까?

Dense(임베딩)와 BM25는 각각 자기만의 점수를 매긴다. 문제는 **두 점수의 척도가 완전히 다르다**는 것.

- Dense cosine 유사도: 보통 `0.3 ~ 0.9` 사이
- BM25 점수: `0 ~ 40+` (문서마다 천차만별)

이 둘을 그냥 더하면 BM25가 항상 이긴다. 정규화를 시도해도 쿼리마다 분포가 달라서 골치 아프다.

### 아이디어: 점수는 버리고, 순위(rank)만 쓰자

RRF의 핵심 발상은 단순하다.

> "각 검색기가 매긴 **점수가 몇 점인지는 무시**하고, **몇 등인지(rank)만** 가지고 합치자."

그리고 "1등한 문서에는 큰 점수, 뒤로 갈수록 빠르게 작아지는 점수"를 주면 된다. 이때 쓰는 함수가 $\frac{1}{k + \text{rank}}$.

### 수식

$$\text{RRF}(d) = \sum_{i} \frac{1}{k + \text{rank}_i(d)}$$

말로 풀면:
- 각 검색기 $i$마다 문서 $d$가 **몇 등**인지 보고 → $\frac{1}{k + \text{등수}}$ 점수를 준다.
- 여러 검색기에서 받은 점수를 **전부 더한다**.
- 최종 점수가 높은 순으로 정렬.

### 숫자로 느껴보기 ($k=60$ 기준)

| 등수 (rank) | 점수 $\frac{1}{60 + \text{rank}}$ |
|---|---|
| 1등 | 1/61 ≈ **0.0164** |
| 2등 | 1/62 ≈ 0.0161 |
| 3등 | 1/63 ≈ 0.0159 |
| 10등 | 1/70 ≈ 0.0143 |
| 100등 | 1/160 ≈ 0.0063 |

포인트 두 가지:
1. **1등과 2등 차이가 아주 작다** → 한 검색기가 1등으로 꼽아도 "압도적 승리"로 치지 않는다. 하지만 **둘 다 상위에 꼽은 문서는 점수가 합산**되어 자연스럽게 올라간다.
2. **뒤로 갈수록 점수가 완만하게 줄어든다** → 10등도 무시당하지 않는다. 한쪽에선 10등이어도 다른 쪽에서 2등이면 충분히 상위로 올라올 수 있다.

상수 $k$ (보통 60)는 "상위권의 영향력을 얼마나 눌러둘지"를 조절한다. $k$가 작으면 1등 독식, $k$가 크면 고루 반영.

### 간단한 예시

Dense가 매긴 순위: `[A, B, C, D]` (A가 1등)
BM25가 매긴 순위: `[C, E, A, F]` (C가 1등)

RRF 점수($k=60$):
- A: $\frac{1}{61}$ (Dense 1등) + $\frac{1}{63}$ (BM25 3등) ≈ **0.0323**
- C: $\frac{1}{63}$ (Dense 3등) + $\frac{1}{61}$ (BM25 1등) ≈ **0.0323**
- B: $\frac{1}{62}$ ≈ 0.0161 (Dense에만 있음)
- E: $\frac{1}{62}$ ≈ 0.0161 (BM25에만 있음)

→ **두 검색기가 공통으로 상위에 올린 A, C**가 자연스럽게 1·2위를 차지한다. 이게 하이브리드 검색의 본질이다.

### 왜 이 방식을 쓰나?

1. **점수 척도 정규화가 필요 없다** (cosine이든 BM25든 순위만 보면 됨).
2. **한 검색기가 이상한 결과를 내놔도** 다른 쪽에서 걸러진다 (outlier에 강함).
3. **구현이 10줄이면 끝**. 튜닝할 하이퍼파라미터도 $k$ 하나뿐.

이 단순함 때문에 TREC 대회부터 엘라스틱서치 공식 가이드까지 표준 기법으로 자리 잡았다.


In [ ]:
def rrf_merge(ranked_lists: list[list[str]], k: int = 60) -> list[tuple[str, float]]:
    '''여러 retriever의 결과(문서 ID 리스트)를 RRF로 병합.'''
    scores: dict[str, float] = {}
    for lst in ranked_lists:
        for rank, doc_id in enumerate(lst, start=1):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

# 장난감 예제
dense_top = ['d3', 'd1', 'd5', 'd2']
bm25_top  = ['d1', 'd4', 'd3', 'd6']
print(rrf_merge([dense_top, bm25_top]))


## 4. LangChain EnsembleRetriever로 하이브리드 구성

`EnsembleRetriever`는 내부적으로 RRF를 사용한다. `weights` 인자로 가중합도 지원한다.

In [ ]:
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain.retrievers import EnsembleRetriever

# Dense retriever
vectordb = FAISS.from_documents(chunks, get_embeddings())
dense = vectordb.as_retriever(search_kwargs={'k': 5})

# Sparse retriever (kiwi 토크나이저 주입)
sparse = BM25Retriever.from_documents(
    chunks,
    preprocess_func=tokenize_ko,  # 한국어 BM25의 핵심
)
sparse.k = 5

# 앙상블 (RRF 가중치 0.5:0.5)
hybrid = EnsembleRetriever(
    retrievers=[dense, sparse],
    weights=[0.5, 0.5],
)
print('하이브리드 retriever 준비 완료')


In [ ]:
query = '개인정보보호법에서 정의하는 개인정보는?'
print('=== Dense only ===')
for d in dense.invoke(query):
    print('-', d.page_content[:70])
print('\n=== BM25 only ===')
for d in sparse.invoke(query):
    print('-', d.page_content[:70])
print('\n=== Hybrid (RRF) ===')
for d in hybrid.invoke(query):
    print('-', d.page_content[:70])


## 5. 가중치 튜닝 팁

| 상황 | 권장 weights (dense, sparse) |
|------|-----------------------------|
| 일반 자연어 질의 위주 | (0.6, 0.4) |
| 법령·약관·제품코드 등 고유명사 多 | (0.3, 0.7) |
| 짧은 키워드 검색 | (0.2, 0.8) |
| 긴 자연어 의도 질의 | (0.7, 0.3) |

실제로는 RAGAS 평가 프레임워크로 `context_recall` 을 최대화하는 가중치를 탐색한다 (RAGAS 공식 문서를 참고).


## 6. 다음 단계: Reranker 얹기

하이브리드 검색은 **recall을 확보**했지만, 상위 20개 후보의 **순서가 최적이 아닐 수 있다**. BM25와 dense가 각자의 기준으로 매긴 순위를 RRF로 단순 결합한 결과라서, "질문에 정말 답을 주는" 문서가 10위에 묻혀 있을 수 있다.

다음 노트북(`02_reranking.ipynb`)에서는 이 hybrid retriever를 **1단계**로 쓰고, 그 위에 **Cross-Encoder Reranker (BGE-reranker-v2-m3)** 를 2단계로 얹어:

- 후보 top-20 → rerank → top-5 로 정제
- "recall은 이미 확보됐다는 전제에서 precision만 끌어올리기"

이것이 Anthropic Contextual Retrieval 논문이 권장하는 실제 운영 파이프라인이다.